# EM Algorithm for Mixture of Gaussians

## Learning Objectives
- Derive the **EM algorithm** as a principled approach to maximise the log-likelihood with latent variables
- Implement the **E-step**: compute soft responsibilities $w_j^{(i)} = p(z^{(i)}=j \mid x^{(i)}; \phi, \mu, \Sigma)$
- Implement the **M-step**: update $\phi_j$, $\mu_j$, $\Sigma_j$ using weighted averages with $w_j^{(i)}$
- Show that the log-likelihood increases monotonically with each EM iteration
- Contrast EM's **soft** cluster assignments with k-means' **hard** assignments

## Problem Statement

We observe $\{x^{(1)}, \ldots, x^{(m)}\}$ drawn from a GMM with latent cluster labels $z^{(i)}$.  
We want to maximise the marginal log-likelihood:
$$\ell(\phi, \mu, \Sigma) = \sum_{i=1}^m \log \sum_{j=1}^k p(x^{(i)} \mid z^{(i)}=j;\, \mu, \Sigma)\, \phi_j$$

**EM algorithm** alternates two steps until convergence:

**E-step** — compute posterior responsibilities (soft assignments):
$$w_j^{(i)} := p(z^{(i)} = j \mid x^{(i)};\, \phi, \mu, \Sigma)
= \frac{p(x^{(i)} \mid z^{(i)}=j;\, \mu, \Sigma)\, \phi_j}
       {\sum_{\ell=1}^k p(x^{(i)} \mid z^{(i)}=\ell;\, \mu, \Sigma)\, \phi_\ell}$$

**M-step** — maximise the expected complete-data log-likelihood:
$$\phi_j := \frac{1}{m}\sum_{i=1}^m w_j^{(i)}, \qquad
\mu_j := \frac{\sum_i w_j^{(i)} x^{(i)}}{\sum_i w_j^{(i)}}, \qquad
\Sigma_j := \frac{\sum_i w_j^{(i)} (x^{(i)}-\mu_j)(x^{(i)}-\mu_j)^T}{\sum_i w_j^{(i)}}$$

**Key property:** $\ell$ increases (or stays the same) at every iteration — EM never decreases the log-likelihood.

## 1. EM Implementation

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal

def gmm_em(X, k, max_iters=100, tol=1e-6, seed=None):
    rng = np.random.default_rng(seed)
    m, n = X.shape

    # Initialise with k-means++ style: pick k random points as means
    idx  = rng.choice(m, k, replace=False)
    mus  = X[idx].copy().astype(float)
    sigs = [np.eye(n) for _ in range(k)]
    phi  = np.full(k, 1.0 / k)

    ll_history = []
    W_history  = []   # soft responsibilities per iteration
    params_history = []

    for _ in range(max_iters):
        # E-step: compute w_j^(i) = p(z^(i)=j | x^(i))
        W = np.zeros((m, k))
        for j in range(k):
            W[:, j] = phi[j] * multivariate_normal(mus[j], sigs[j]).pdf(X)
        row_sums = W.sum(axis=1, keepdims=True)
        W /= (row_sums + 1e-300)

        # Log-likelihood
        ll = np.sum(np.log(row_sums.squeeze() + 1e-300))
        ll_history.append(ll)
        W_history.append(W.copy())
        params_history.append((mus.copy(), [s.copy() for s in sigs], phi.copy()))

        # M-step: update phi, mu, Sigma
        Wsum = W.sum(axis=0)   # (k,)
        phi  = Wsum / m
        mus  = (W.T @ X) / Wsum[:, None]   # (k, n)
        for j in range(k):
            diff = X - mus[j]               # (m, n)
            sigs[j] = (W[:, j:j+1] * diff).T @ diff / Wsum[j]

        if len(ll_history) >= 2 and abs(ll_history[-1] - ll_history[-2]) < tol:
            break

    return mus, sigs, phi, W, ll_history, W_history, params_history


# Generate data from a known GMM
np.random.seed(42)
k_true = 3
phi_true  = np.array([0.4, 0.35, 0.25])
mus_true  = np.array([[0.0, 0.0], [4.0, 1.0], [2.0, 4.5]])
sigs_true = [
    np.array([[1.0, 0.5], [0.5, 0.8]]),
    np.array([[0.8, -0.3], [-0.3, 1.2]]),
    np.array([[1.5, 0.0], [0.0, 0.5]]),
]
m = 200
z_true = np.random.choice(k_true, size=m, p=phi_true)
X = np.array([np.random.multivariate_normal(mus_true[z], sigs_true[z]) for z in z_true])

mus_em, sigs_em, phi_em, W_final, ll_hist, W_hist, params_hist = gmm_em(X, k=3, seed=7)

print(f'Converged in {len(ll_hist)} iterations')
print(f'Final log-likelihood: {ll_hist[-1]:.2f}')
print('\nEstimated phi:', phi_em.round(3))
print('True phi:     ', phi_true)
print('\nEstimated means:\n', mus_em.round(2))
print('True means:\n', mus_true)

## 2. Log-Likelihood Convergence

EM is guaranteed to not decrease $\ell$ at each iteration.  
In practice the log-likelihood increases rapidly in the first few iterations then plateaus.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: log-likelihood vs iteration
ax = axes[0]
ax.plot(range(1, len(ll_hist)+1), ll_hist, 'b-o', lw=2.5, ms=6)
ax.set_xlabel('EM Iteration')
ax.set_ylabel('Log-likelihood $\\ell$')
ax.set_title('Log-likelihood increases monotonically\n(EM guarantee: $\\ell$ never decreases)')
ax.text(0.55, 0.15,
        f'Initial $\\ell$: {ll_hist[0]:.1f}\nFinal $\\ell$: {ll_hist[-1]:.1f}\n'
        f'Iterations: {len(ll_hist)}',
        transform=ax.transAxes, fontsize=9, va='bottom',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

# Right: change in log-likelihood per iteration (semi-log)
ax = axes[1]
deltas = np.diff(ll_hist)
ax.semilogy(range(1, len(deltas)+1), deltas + 1e-10, 'r-o', lw=2.5, ms=6)
ax.set_xlabel('EM Iteration')
ax.set_ylabel('$\\Delta \\ell$ (log scale)')
ax.set_title('Improvement per iteration decreases\n(exponential convergence near optimum)')

fig.suptitle('EM Algorithm — Log-Likelihood Convergence', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Soft Assignments Evolving Over Iterations

In the E-step each point receives a **soft** responsibility vector $w^{(i)} \in \mathbb{R}^k$ summing to 1.  
Early in training these are diffuse; as parameters improve they sharpen toward one cluster.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

colors = ['#2166ac', '#d6604d', '#4dac26']

# Show snapshots at iterations 1, 2, 3, and final
snap_iters = [0, 1, 2, len(W_hist)-1]
snap_labels = [f'Iteration {i+1}' for i in snap_iters[:-1]] + [f'Final (iter {len(W_hist)})']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

xg = np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 150)
yg = np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 150)
Xg, Yg = np.meshgrid(xg, yg)
pos = np.dstack([Xg, Yg])

for ax, it, label in zip(axes, snap_iters, snap_labels):
    W_snap = W_hist[it]
    mus_snap, sigs_snap, _ = params_hist[it]

    # Colour each point by its dominant responsibility, alpha by confidence
    dominant = W_snap.argmax(axis=1)
    confidence = W_snap.max(axis=1)   # max weight → how sure EM is

    for j in range(3):
        mask = dominant == j
        ax.scatter(X[mask, 0], X[mask, 1],
                   c=colors[j], s=20, alpha=confidence[mask] * 0.9 + 0.1)

        # Contour for current estimated Gaussian
        try:
            Z_pdf = multivariate_normal(mus_snap[j], sigs_snap[j]).pdf(pos)
            ax.contour(Xg, Yg, Z_pdf, levels=3, colors=[colors[j]], alpha=0.6, linewidths=1.5)
        except Exception:
            pass
        ax.scatter(*mus_snap[j], marker='X', s=180, c=colors[j],
                   edgecolors='k', lw=1.5, zorder=5)

    ax.set_title(f'{label}\n$\\ell = {ll_hist[it]:.1f}$', fontsize=9)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

fig.suptitle('EM Soft Assignments Over Iterations (colour = dominant cluster, brightness = confidence)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. EM (Soft) vs. K-Means (Hard) Assignments

| Property | EM / GMM | K-Means |
|---|---|---|
| Assignment type | **Soft**: $w_j^{(i)} \in [0,1]$, sums to 1 | **Hard**: $c^{(i)} \in \{1,\ldots,k\}$ |
| Covariance | Full $\Sigma_j$ per cluster | Implicitly spherical (distance = norm²) |
| Objective | Marginal log-likelihood $\ell$ | Distortion $J(c, \mu)$ |
| Convergence | $\ell$ monotonically non-decreasing | $J$ monotonically non-increasing |
| Uncertainty | Captures overlap between clusters | Cannot express uncertainty |

K-means is the hard-assignment limiting case of EM when all $\Sigma_j = \sigma^2 I$ and $\sigma^2 \to 0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

# Run k-means on same data for comparison
def kmeans(X, k, max_iters=100, seed=None):
    rng = np.random.default_rng(seed)
    m   = X.shape[0]
    mu  = X[rng.choice(m, k, replace=False)].copy().astype(float)
    for _ in range(max_iters):
        dists = np.linalg.norm(X[:, None, :] - mu[None, :, :], axis=2)
        c     = np.argmin(dists, axis=1)
        mu_new = np.array([X[c == j].mean(axis=0) if (c == j).any() else mu[j] for j in range(k)])
        if np.allclose(mu, mu_new):
            break
        mu = mu_new
    return mu, c

mu_km, c_km = kmeans(X, k=3, seed=7)

colors = ['#2166ac', '#d6604d', '#4dac26']

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))

xg = np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 150)
yg = np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 150)
Xg, Yg = np.meshgrid(xg, yg)
pos = np.dstack([Xg, Yg])

# Left: true labels
ax = axes[0]
for j in range(3):
    pts = X[z_true == j]
    ax.scatter(pts[:, 0], pts[:, 1], c=colors[j], s=25, alpha=0.7, label=f'True $z={j+1}$')
    ax.scatter(*mus_true[j], marker='X', s=200, c=colors[j], edgecolors='k', lw=1.5, zorder=5)
ax.set_title('True GMM labels\n(oracle)', fontsize=10)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend(fontsize=8)

# Middle: k-means hard assignments
ax = axes[1]
for j in range(3):
    pts = X[c_km == j]
    ax.scatter(pts[:, 0], pts[:, 1], c=colors[j], s=25, alpha=0.7)
    ax.scatter(*mu_km[j], marker='X', s=200, c=colors[j], edgecolors='k', lw=1.5, zorder=5)
ax.set_title('K-Means: hard assignments\n(one colour per point)', fontsize=10)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

# Right: EM soft assignments (colour blended by responsibilities)
ax = axes[2]
# Blend RGB colours by w_j^(i)
rgb_colors = np.array([[33, 102, 172], [214, 96, 77], [77, 172, 38]], dtype=float) / 255.0
blended = W_final @ rgb_colors   # (m, 3)
ax.scatter(X[:, 0], X[:, 1], c=blended, s=25, alpha=0.85)
for j in range(3):
    ax.scatter(*mus_em[j], marker='X', s=200, c=colors[j], edgecolors='k', lw=1.5, zorder=5)
    try:
        Z_pdf = multivariate_normal(mus_em[j], sigs_em[j]).pdf(pos)
        ax.contour(Xg, Yg, Z_pdf, levels=3, colors=[colors[j]], alpha=0.5, linewidths=1.5)
    except Exception:
        pass
ax.set_title('EM (GMM): soft assignments\n(colour = blended responsibilities)', fontsize=10)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

fig.suptitle('EM Soft vs. K-Means Hard Cluster Assignments',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Soft assignment entropy — lower entropy = more confident assignments
entropy = -np.sum(W_final * np.log(W_final + 1e-300), axis=1)
print(f'Mean soft-assignment entropy: {entropy.mean():.3f}  (0 = perfectly confident)')
print(f'Fraction of points with dominant weight > 0.95: {(W_final.max(axis=1) > 0.95).mean():.1%}')

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="374"
     viewBox="0 0 780 374" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>

  <!-- Row 1: Initial params -->
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Initialise</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">&#x3c6;, {&#x3bc;&#x2c7;}, {&#x3a3;&#x2c7;}</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >random initialisation; result may vary &#x2014; multiple restarts recommended</text>

  <!-- step 1&#x2192;2 -->
  <line x1="102" y1="58" x2="102" y2="108"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="82" font-size="11.5" font-style="italic" fill="#555"
        >E-step: w&#x2c7;&#x2071; = p(z&#x2071;=j|x&#x2071;) by Bayes&#x2019; rule</text>

  <!-- Row 2: E-step -->
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="140" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">E-step</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >soft responsibilities w&#x2c7;&#x2071; &#x2208; [0,1], &#x2211;&#x2c7; w&#x2c7;&#x2071; = 1 &#x2014; replace hard indicator with expectation</text>

  <!-- step 2&#x2192;3 -->
  <line x1="102" y1="158" x2="102" y2="208"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="182" font-size="11.5" font-style="italic" fill="#555"
        >M-step: maximise E[complete-data log-likelihood]</text>

  <!-- Row 3: M-step -->
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="240" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">M-step</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >weighted MLE: &#x3c6;&#x2c7;, &#x3bc;&#x2c7;, &#x3a3;&#x2c7; update using w&#x2c7;&#x2071; as weights &#x2014; closed form</text>

  <!-- step 3&#x2192;4 -->
  <line x1="102" y1="258" x2="102" y2="308"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="282" font-size="11.5" font-style="italic" fill="#555"
        >&#x2113; increases each iteration &#x2014; repeat until convergence</text>

  <!-- Row 4: Converged GMM -->
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="340" font-size="13.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Converged GMM</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >local optimum of marginal &#x2113; &#x2014; Jensen&#x2019;s ineq. guarantees &#x2113; non-decreasing (next notebooks)</text>
</svg>
""")

## Summary

| Concept | Formula | Key Insight |
|---|---|---|
| E-step | $w_j^{(i)} = \frac{\phi_j \mathcal{N}(x^{(i)};\mu_j,\Sigma_j)}{\sum_\ell \phi_\ell \mathcal{N}(x^{(i)};\mu_\ell,\Sigma_\ell)}$ | Soft responsibility; posterior of $z^{(i)}$ given $x^{(i)}$ |
| M-step $\phi_j$ | $\phi_j := \frac{1}{m}\sum_i w_j^{(i)}$ | Weighted fraction of points in cluster $j$ |
| M-step $\mu_j$ | $\mu_j := \frac{\sum_i w_j^{(i)} x^{(i)}}{\sum_i w_j^{(i)}}$ | Weighted mean of points in cluster $j$ |
| M-step $\Sigma_j$ | $\Sigma_j := \frac{\sum_i w_j^{(i)}(x^{(i)}-\mu_j)(x^{(i)}-\mu_j)^T}{\sum_i w_j^{(i)}}$ | Weighted covariance |
| Convergence | $\ell$ monotonically non-decreasing | Guaranteed by Jensen's inequality (see next notebooks) |
| vs. k-means | Soft vs. hard assignments | k-means is the $\sigma^2 \to 0$ limiting case |

**Key insight:** EM replaces the intractable MLE (optimising over a log-of-sum) with an alternating two-step procedure; the E-step computes soft assignments and the M-step updates parameters with weighted closed-form formulas — the log-likelihood is guaranteed never to decrease.